# Wordle 各档比例的联合预测

参考 [2023 MCM C](https://www.contest.comap.com/undergraduate/contests/mcm/contests/2023/problems/)。
使用模拟的每日档位计数，输出 1 至 6 次猜中及失败的七档比例。
给定单词难度类别和报告总数，对比多项分布与 Dirichlet–Multinomial。
后者允许同一类单词的概率组成在不同日期之间波动。

In [1]:
import numpy as np
import pandas as pd
import pymc as pm
from scipy.optimize import minimize_scalar
from scipy.special import gammaln

rng = np.random.default_rng(2023)
categories = ["1", "2", "3", "4", "5", "6", "X"]
average = np.array([
    [0.02, 0.14, 0.38, 0.28, 0.13, 0.04, 0.01],
    [0.01, 0.05, 0.25, 0.36, 0.21, 0.09, 0.03],
    [0.01, 0.03, 0.13, 0.31, 0.28, 0.16, 0.08],
])
difficulty = rng.integers(0, len(average), size=180)
totals = rng.integers(300, 850, size=180)
observed = np.vstack([
    rng.multinomial(total, rng.dirichlet(60 * average[group]))
    for group, total in zip(difficulty, totals)
])
split = 140
group_counts = np.vstack([
    observed[:split][difficulty[:split] == group].sum(axis=0) + 0.5
    for group in range(len(average))
])
centers = group_counts / group_counts.sum(axis=1, keepdims=True)
train_probs = centers[difficulty[:split]]
train_counts, train_totals = observed[:split], totals[:split]

def negative_log_likelihood(log_concentration):
    alpha = np.exp(log_concentration) * train_probs
    concentration = alpha.sum(axis=1)
    log_kernel = (
        gammaln(concentration) - gammaln(concentration + train_totals)
        + (gammaln(alpha + train_counts) - gammaln(alpha)).sum(axis=1)
    )
    # The multinomial coefficient is constant as concentration changes.
    return -log_kernel.sum()

result = minimize_scalar(negative_log_likelihood, bounds=(-3, 9), method="bounded")
assert result.success
concentration = np.exp(result.x)
print(f"Fitted concentration: {concentration:.2f}")

Fitted concentration: 62.64


In [2]:
test_totals = totals[split:]
test_counts = observed[split:]
test_probs = centers[difficulty[split:]]
distributions = {
    "Multinomial": pm.Multinomial.dist(n=test_totals, p=test_probs),
    "Dirichlet-Multinomial": pm.DirichletMultinomial.dist(
        n=test_totals, a=concentration * test_probs
    ),
}

comparison = {}
all_intervals = {}
for label, distribution in distributions.items():
    log_probability = pm.logp(distribution, test_counts).eval()
    draws = pm.draw(distribution, draws=350, random_seed=2023)
    assert np.all(draws.sum(axis=-1) == test_totals)
    proportions = draws / test_totals[None, :, None]
    lower, upper = np.quantile(proportions, [0.05, 0.95], axis=0)
    actual = test_counts / test_totals[:, None]
    comparison[label] = {
        "negative log score per word": -log_probability.mean(),
        "marginal coverage": ((actual >= lower) & (actual <= upper)).mean(),
        "mean interval width": (upper - lower).mean(),
    }
    all_intervals[label] = lower, upper

pd.DataFrame(comparison).T.round(3)

,negative log score per word,marginal coverage,mean interval width
Multinomial,51.675,0.346,0.041
Dirichlet-Multinomial,25.360,0.857,0.128


In [3]:
lower, upper = all_intervals["Dirichlet-Multinomial"]
pd.DataFrame({
    "predicted %": 100 * test_probs[0],
    "observed %": 100 * test_counts[0] / test_totals[0],
    "90% lower": 100 * lower[0],
    "90% upper": 100 * upper[0],
}, index=categories).round(2)

,predicted %,observed %,90% lower,90% upper
1,0.88,0.84,0.00,3.56
2,3.67,5.86,0.56,8.23
3,12.03,16.46,6.12,19.53
4,32.09,23.85,21.96,41.92
5,26.93,40.73,17.71,36.40
6,15.75,9.62,8.85,24.07
X,8.65,2.65,3.35,16.36


每次联合模拟的七档比例都会加总到 100%。表中上下限是各档分别计算的边际区间。
应用到真实数据时，可从词频、重复字母等赛前可知特征估计概率中心。
如果数据只有四舍五入后的百分比，先明确可否获得各档计数和分母，再选择似然。